# Analyse Exploratoire Des Donnees (EDA)

Dans cette phase d'EDA, l'objectif est d'abord de comprendre le contenu de chaque fichier `.csv` mis a disposition : quelle table il répresente, àa quel niveau d'observation elle se situe, quelles variables elle contient, et comment elle peut se "rattacher" aux autres sources de données. Ce travail permet d'identifier la structure globale du jeu de données et de préparer une stratégie de fusion cohèrente autour de la table principale.

Une fois cette cartographie réalisée, l'analyse consiste à déterminer comment assembler les differentes tables de facon pertinente, puis à evaluer la qualité des données obtenues. Cela inclût notamment la gestion des doublons, l'étude et le traitement des valeurs manquantes, la détection d'éventuelles valeurs abérrantes, ainsi qu'une première vérification de la cohérence globale des variables.

Cette étape est essentielle pour construire une base de travail propre, exploitable et robuste pour les analyses et la modélisation à venir.


In [2]:
from pathlib import Path
import sys
import importlib
import numpy as np

# Détecte automatiquement la racine projet (celle qui contient /src)
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.Fonctions_EDA as Fonctions_EDA
importlib.reload(Fonctions_EDA)
from src.Fonctions_EDA import eda_overview

In [3]:
import pandas as pd
from pathlib import Path
import importlib
import math
import matplotlib.pyplot as plt
import seaborn as sns


 
base_path = Path("../data/raw")

df_app_test = pd.read_csv(base_path / "application_test.csv")
df_app_train = pd.read_csv(base_path / "application_train.csv")
df_bureau_balance = pd.read_csv(base_path / "bureau_balance.csv")
df_bureau = pd.read_csv(base_path / "bureau.csv")
df_credit_card_balance = pd.read_csv(base_path / "credit_card_balance.csv")
df_hc_columns_description = pd.read_csv(base_path / "HomeCredit_columns_description.csv",encoding="cp1252")
df_install_payments = pd.read_csv(base_path / "installments_payments.csv")
df_pos_cash_balance = pd.read_csv(base_path / "POS_CASH_balance.csv")
df_previous_application = pd.read_csv(base_path / "previous_application.csv")
df_sample_submission = pd.read_csv(base_path / "sample_submission.csv")



In [21]:
print(df_app_test.shape)
print(df_app_train.shape)
print(df_bureau_balance.shape)
print(df_bureau.shape)
print(df_credit_card_balance.shape)
print(df_hc_columns_description.shape)
print(df_install_payments.shape)
print(df_pos_cash_balance.shape)
print(df_previous_application.shape)
print(df_sample_submission.shape)

(48744, 121)
(307511, 122)
(27299925, 3)
(1716428, 17)
(3840312, 23)
(219, 5)
(13605401, 8)
(10001358, 8)
(1670214, 37)
(48744, 2)


## Identification des clés et de la granularité des tables

Les dimensions des différents fichiers ne sont pas censées être homogènes, car chaque table décrit un niveau d'information différent. Certaines contiennent une ligne par client, tandis que d'autres recensent plusieurs événements historiques pour un même client ou pour un même crédit.

L'objectif n'est donc pas d'obtenir des tables de même longueur, mais d'identifier pour chacune :
- sa granularité d'observation ;
- sa clé primaire potentielle ;
- sa ou ses clés étrangères ;
- son lien avec la table principale.

Cette étape est indispensable pour construire une stratégie de jointure cohérente et éviter des duplications massives lors des fusions.

In [10]:
dfs = {
    "df_app_test": df_app_test,
    "df_app_train": df_app_train,
    "df_bureau_balance": df_bureau_balance,
    "df_bureau": df_bureau,
    "df_credit_card_balance": df_credit_card_balance,
    "df_hc_columns_description": df_hc_columns_description,
    "df_install_payments": df_install_payments,
    "df_pos_cash_balance": df_pos_cash_balance,
    "df_previous_application": df_previous_application,
    "df_sample_submission": df_sample_submission,
}


for name, df in dfs.items():
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("columns clés potentielles:", [c for c in df.columns if "SK_ID" in c])



--- df_app_test ---
shape: (48744, 121)
columns clés potentielles: ['SK_ID_CURR']

--- df_app_train ---
shape: (307511, 122)
columns clés potentielles: ['SK_ID_CURR']

--- df_bureau_balance ---
shape: (27299925, 3)
columns clés potentielles: ['SK_ID_BUREAU']

--- df_bureau ---
shape: (1716428, 17)
columns clés potentielles: ['SK_ID_CURR', 'SK_ID_BUREAU']

--- df_credit_card_balance ---
shape: (3840312, 23)
columns clés potentielles: ['SK_ID_PREV', 'SK_ID_CURR']

--- df_hc_columns_description ---
shape: (219, 5)
columns clés potentielles: []

--- df_install_payments ---
shape: (13605401, 8)
columns clés potentielles: ['SK_ID_PREV', 'SK_ID_CURR']

--- df_pos_cash_balance ---
shape: (10001358, 8)
columns clés potentielles: ['SK_ID_PREV', 'SK_ID_CURR']

--- df_previous_application ---
shape: (1670214, 37)
columns clés potentielles: ['SK_ID_PREV', 'SK_ID_CURR']

--- df_sample_submission ---
shape: (48744, 2)
columns clés potentielles: ['SK_ID_CURR']


## Structure relationnelle du dataset

Le dataset Home Credit est organisé autour d'une table principale, `application_train` / `application_test`, qui contient une ligne par demande de prêt. Cette table constitue la base centrale de l'analyse. La variable cible `TARGET` n'est présente que dans `application_train`.

Autour de cette table principale, plusieurs tables secondaires décrivent l'historique financier des clients, soit auprès d'autres organismes, soit auprès de Home Credit lui-même. La table `bureau` recense les crédits externes signalés au bureau de crédit, avec un lien vers la table principale via `SK_ID_CURR`. La table `bureau_balance` apporte ensuite un historique mensuel de ces crédits via `SK_ID_BUREAU`.

Du côté des crédits historiques chez Home Credit, `previous_application` contient une ligne par demande précédente du client. Cette table est reliée à la table principale par `SK_ID_CURR`, et sert de point d'entrée vers plusieurs tables de détail : `POS_CASH_balance`, `installments_payments` et `credit_card_balance`, généralement reliées par `SK_ID_PREV`.

Ainsi, les tables ne sont pas au même niveau de granularité :
- `application_train` / `application_test` : une ligne par client ou dossier
- `bureau` : une ligne par crédit externe
- `bureau_balance` : une ligne par mois et par crédit externe
- `previous_application` : une ligne par demande précédente
- `POS_CASH_balance` : une ligne par mois et par crédit précédent
- `installments_payments` : une ligne par paiement ou échéance
- `credit_card_balance` : une ligne par mois et par carte / crédit précédent

Cette structure implique qu'une fusion directe de toutes les tables provoquerait des duplications massives. La bonne approche consiste d'abord à agréger les tables secondaires à leur bon niveau, puis à les rattacher progressivement à la table principale.


## Choix d'un traitement sous PostgreSQL

Compte tenu du volume important de certaines tables, un traitement exclusivement en mémoire avec `pandas` devient rapidement limité, notamment pour les jointures, les agrégations et les contrôles de qualité de données. L'exploration initiale dans Python reste utile pour comprendre la structure des fichiers, mais le travail principal de préparation et de consolidation des données sera plus robuste dans une base PostgreSQL.

L'utilisation de PostgreSQL permet de charger les fichiers dans des tables relationnelles, d'identifier plus facilement les clés de jointure, de contrôler les cardinalités, d'exécuter des agrégations intermédiaires volumineuses et de préparer une table d'analyse finale de manière plus fiable. Cette approche est particulièrement adaptée à un dataset structuré en plusieurs tables liées entre elles, avec des historiques mensuels et transactionnels très détaillés.


In [11]:
import os
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("PGHOST"),
    "port": int(os.getenv("PGPORT", 5432)),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}

DB_NAME = os.getenv("PGDATABASE", "home_credit")

conn = psycopg2.connect(dbname="postgres", **DB_CONFIG)
conn.autocommit = True
cur = conn.cursor()

cur.execute(
    "SELECT 1 FROM pg_database WHERE datname = %s",
    (DB_NAME,)
)

if cur.fetchone() is None:
    cur.execute(
        sql.SQL("CREATE DATABASE {}").format(
            sql.Identifier(DB_NAME)
        )
    )
    print(f"Base '{DB_NAME}' créée.")
else:
    print(f"Base '{DB_NAME}' existe déjà.")

cur.close()
conn.close()


Base 'home_credit' existe déjà.


In [1]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd().resolve().parent
scripts_dir = project_root / "scripts"

subprocess.run([sys.executable, str(scripts_dir / "create_db.py")], check=True)
subprocess.run([sys.executable, str(scripts_dir / "create_tables.py")], check=True)
subprocess.run([sys.executable, str(scripts_dir / "load_data.py")], check=True)


CalledProcessError: Command '['c:\\Users\\kevin\\Documents\\P6\\P6_MLOps_1-2\\.venv\\Scripts\\python.exe', 'C:\\Users\\kevin\\Documents\\P6\\P6_MLOps_1-2\\scripts\\load_data.py']' returned non-zero exit status 1.

In [16]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.Fonctions_EDA import eda_overview

eda_overview(df_app_train)


## 1) Vue globale


,n_lignes,n_colonnes,doublons,%_doublons
0,307511,122,0,0.0



## 2) Qualité des colonnes et cardinalité


,dtype,%null,non_null,null,n_uniques
COMMONAREA_AVG,float64,69.87,92646,214865,3181
COMMONAREA_MODE,float64,69.87,92646,214865,3128
COMMONAREA_MEDI,float64,69.87,92646,214865,3202
NONLIVINGAPARTMENTS_AVG,float64,69.43,93997,213514,386
NONLIVINGAPARTMENTS_MODE,float64,69.43,93997,213514,167
NONLIVINGAPARTMENTS_MEDI,float64,69.43,93997,213514,214
LIVINGAPARTMENTS_AVG,float64,68.35,97312,210199,1868
LIVINGAPARTMENTS_MODE,float64,68.35,97312,210199,736
LIVINGAPARTMENTS_MEDI,float64,68.35,97312,210199,1097
FLOORSMIN_AVG,float64,67.85,98869,208642,305



## 3) Variables numériques (106)

### 3.1) Complétude variables numériques


...


### 3.2) Distribution générale variables numériques


,%outliers (IQR),nb_outliers_bas,nb_outliers_haut,min,Q1,median,Q3,max,skew,kurtosis
REGION_RATING_CLIENT,26.19,32197,48330,1.000000e+00,2.000000,2.000000,2.000000,3.000000e+00,0.087468,0.800416
REGION_RATING_CLIENT_W_CITY,25.37,34167,43860,1.000000e+00,2.000000,2.000000,2.000000,3.000000e+00,0.059730,0.933584
DAYS_EMPLOYED,23.48,16843,55374,-1.791200e+04,-2760.000000,-1213.000000,-289.000000,3.652430e+05,1.664346,0.771612
REG_CITY_NOT_WORK_CITY,23.05,0,70867,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00,1.280138,-0.361250
FLAG_WORK_PHONE,19.94,0,61308,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00,1.504950,0.264876
AMT_REQ_CREDIT_BUREAU_QRT,19.01,0,50575,0.000000e+00,0.000000,0.000000,0.000000,2.610000e+02,134.365776,43707.464752
FLAG_EMP_PHONE,18.01,55386,0,0.000000e+00,1.000000,1.000000,1.000000,1.000000e+00,-1.664886,0.771852
LIVE_CITY_NOT_WORK_CITY,17.96,0,55215,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00,1.669795,0.788220
NONLIVINGAPARTMENTS_AVG,16.57,0,15580,0.000000e+00,0.000000,0.000000,0.003900,1.000000e+00,15.541185,284.730336
AMT_REQ_CREDIT_BUREAU_MON,16.45,0,43759,0.000000e+00,0.000000,0.000000,0.000000,2.700000e+01,7.804848,90.434857



## 4) Variables catégorielles (16)


,n_uniques,null,%null,modalite_top,nb_modalite_top
FONDKAPREMONT_MODE,4,210295,68.39,reg oper account,73830
WALLSMATERIAL_MODE,7,156341,50.84,Panel,66040
HOUSETYPE_MODE,3,154297,50.18,block of flats,150503
EMERGENCYSTATE_MODE,2,145755,47.40,No,159428
OCCUPATION_TYPE,18,96391,31.35,Laborers,55186
NAME_TYPE_SUITE,7,1292,0.42,Unaccompanied,248526
ORGANIZATION_TYPE,58,0,0.00,Business Entity Type 3,67992
NAME_INCOME_TYPE,8,0,0.00,Working,158774
WEEKDAY_APPR_PROCESS_START,7,0,0.00,TUESDAY,53901
NAME_FAMILY_STATUS,6,0,0.00,Married,196432



## 5) Variables temporelles (0)

Aucune variable datetime.


## Etape 0 - Preparation avant l'EDA

Avant d'entrer dans l'analyse d?taill?e, on r?alise un mini-audit global des tables : volum?trie, cl?s disponibles, grain suppos?, relations et premiers indicateurs de qualit?.


In [ ]:
table_inventory = pd.DataFrame([
    {'table_name': 'application_train', 'n_rows': len(df_app_train), 'n_cols': df_app_train.shape[1], 'keys_available': 'SK_ID_CURR', 'grain_suppose': '1 ligne par demande actuelle / client', 'has_target': True},
    {'table_name': 'application_test', 'n_rows': len(df_app_test), 'n_cols': df_app_test.shape[1], 'keys_available': 'SK_ID_CURR', 'grain_suppose': '1 ligne par demande actuelle / client', 'has_target': False},
    {'table_name': 'bureau', 'n_rows': len(df_bureau), 'n_cols': df_bureau.shape[1], 'keys_available': 'SK_ID_CURR, SK_ID_BUREAU', 'grain_suppose': '1 ligne par cr?dit bureau', 'has_target': False},
    {'table_name': 'bureau_balance', 'n_rows': len(df_bureau_balance), 'n_cols': df_bureau_balance.shape[1], 'keys_available': 'SK_ID_BUREAU', 'grain_suppose': '1 ligne par mois et cr?dit bureau', 'has_target': False},
    {'table_name': 'previous_application', 'n_rows': len(df_previous_application), 'n_cols': df_previous_application.shape[1], 'keys_available': 'SK_ID_PREV, SK_ID_CURR', 'grain_suppose': '1 ligne par ancienne demande', 'has_target': False},
    {'table_name': 'POS_CASH_balance', 'n_rows': len(df_pos_cash_balance), 'n_cols': df_pos_cash_balance.shape[1], 'keys_available': 'SK_ID_PREV, SK_ID_CURR', 'grain_suppose': '1 ligne par mois pour un ancien cr?dit POS/cash', 'has_target': False},
    {'table_name': 'credit_card_balance', 'n_rows': len(df_credit_card_balance), 'n_cols': df_credit_card_balance.shape[1], 'keys_available': 'SK_ID_PREV, SK_ID_CURR', 'grain_suppose': '1 ligne par mois pour un ancien cr?dit carte', 'has_target': False},
    {'table_name': 'installments_payments', 'n_rows': len(df_install_payments), 'n_cols': df_install_payments.shape[1], 'keys_available': 'SK_ID_PREV, SK_ID_CURR', 'grain_suppose': '1 ligne par paiement observ?', 'has_target': False},
])
display(table_inventory.sort_values('n_rows', ascending=False))


### Cartographie relationnelle simplifiee

- `application_train` / `application_test` : table centrale, grain = 1 ligne par client courant
- `bureau` : plusieurs cr?dits bureau par client
- `bureau_balance` : plusieurs mois par cr?dit bureau
- `previous_application` : plusieurs demandes pr?c?dentes par client
- `POS_CASH_balance` : plusieurs mois par ancien cr?dit
- `credit_card_balance` : plusieurs mois par ancien cr?dit carte
- `installments_payments` : plusieurs paiements par ancien cr?dit


In [ ]:
def quick_table_audit(df: pd.DataFrame, table_name: str, keys: list[str] | None = None) -> dict:
    keys = keys or []
    n_rows, n_cols = df.shape
    numeric_cols = df.select_dtypes(include=['number']).columns
    cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
    missing_pct = (df.isna().mean() * 100).round(2)
    quasi_constant_cols = []
    for col in df.columns:
        top_freq = df[col].value_counts(dropna=False, normalize=True)
        if not top_freq.empty and top_freq.iloc[0] >= 0.95:
            quasi_constant_cols.append(col)

    key_uniqueness = {}
    for key in keys:
        if key in df.columns:
            key_uniqueness[key] = bool(df[key].is_unique)

    return {
        'table_name': table_name,
        'n_rows': n_rows,
        'n_cols': n_cols,
        'exact_duplicates': int(df.duplicated().sum()),
        'n_numeric_cols': len(numeric_cols),
        'n_categorical_cols': len(cat_cols),
        'avg_missing_pct': round(float(missing_pct.mean()), 2),
        'max_missing_pct': round(float(missing_pct.max()), 2),
        'n_quasi_constant_cols': len(quasi_constant_cols),
        'key_uniqueness': key_uniqueness,
    }


In [ ]:
audit_summary = pd.DataFrame([
    quick_table_audit(df_app_train, 'application_train', ['SK_ID_CURR']),
    quick_table_audit(df_app_test, 'application_test', ['SK_ID_CURR']),
    quick_table_audit(df_bureau, 'bureau', ['SK_ID_BUREAU']),
    quick_table_audit(df_bureau_balance, 'bureau_balance', ['SK_ID_BUREAU']),
    quick_table_audit(df_previous_application, 'previous_application', ['SK_ID_PREV']),
    quick_table_audit(df_pos_cash_balance, 'POS_CASH_balance', ['SK_ID_PREV']),
    quick_table_audit(df_credit_card_balance, 'credit_card_balance', ['SK_ID_PREV']),
    quick_table_audit(df_install_payments, 'installments_payments', []),
])
display(audit_summary.sort_values('max_missing_pct', ascending=False))


## Etape 1 - EDA du dataset principal `application_train`

Objectif : comprendre le profil du client courant, la cible `TARGET`, les variables les plus prometteuses et les premi?res features d?riv?es envisageables.


In [ ]:
app_train_type_summary = pd.DataFrame({
    'dtype': df_app_train.dtypes.astype(str),
    'missing_pct': (df_app_train.isna().mean() * 100).round(2),
    'n_unique': df_app_train.nunique(dropna=True),
})
app_train_type_summary['family'] = 'categorical_or_other'
app_train_type_summary.loc[df_app_train.select_dtypes(include=['number']).columns, 'family'] = 'numeric'
app_train_type_summary.loc[app_train_type_summary['n_unique'] <= 2, 'family'] = 'binary_or_quasi_binary'
display(app_train_type_summary.sort_values(['family', 'missing_pct'], ascending=[True, False]))

type_mix = app_train_type_summary['family'].value_counts().rename_axis('family').reset_index(name='n_columns')
display(type_mix)


In [ ]:
target_summary = (
    df_app_train['TARGET'].value_counts(dropna=False)
    .rename_axis('TARGET')
    .reset_index(name='nb_clients')
)
target_summary['pct_clients'] = (target_summary['nb_clients'] / len(df_app_train) * 100).round(2)
baseline_majoritaire = round(target_summary['pct_clients'].max(), 2)
display(target_summary)
print(f'Baseline na?ve (classe majoritaire) : {baseline_majoritaire}%')


In [ ]:
df_app_train_features = df_app_train.copy()
df_app_train_features['AGE_YEARS'] = (-df_app_train_features['DAYS_BIRTH'] / 365.25).round(1)
df_app_train_features['YEARS_EMPLOYED'] = (-df_app_train_features['DAYS_EMPLOYED'] / 365.25).round(1)
df_app_train_features['EMPLOYED_SENTINEL_365243'] = (df_app_train_features['DAYS_EMPLOYED'] == 365243).astype(int)
df_app_train_features['CREDIT_INCOME_RATIO'] = df_app_train_features['AMT_CREDIT'] / df_app_train_features['AMT_INCOME_TOTAL']
df_app_train_features['ANNUITY_INCOME_RATIO'] = df_app_train_features['AMT_ANNUITY'] / df_app_train_features['AMT_INCOME_TOTAL']
df_app_train_features['GOODS_CREDIT_RATIO'] = df_app_train_features['AMT_GOODS_PRICE'] / df_app_train_features['AMT_CREDIT']
df_app_train_features['EXT_SOURCE_MEAN'] = df_app_train_features[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df_app_train_features['EXT_SOURCE_MIN'] = df_app_train_features[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
df_app_train_features['EXT_SOURCE_MAX'] = df_app_train_features[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)
df_app_train_features['NB_DOCS_PROVIDED'] = df_app_train_features.filter(regex=r'^FLAG_DOCUMENT_').sum(axis=1)


In [ ]:
focus_numeric = [
    'AGE_YEARS', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'GOODS_CREDIT_RATIO',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_MEAN',
    'DAYS_EMPLOYED', 'YEARS_EMPLOYED', 'NB_DOCS_PROVIDED'
]
numeric_by_target = df_app_train_features.groupby('TARGET')[focus_numeric].agg(['mean', 'median']).round(2)
display(numeric_by_target)


In [ ]:
sentinel_share = (df_app_train_features['EMPLOYED_SENTINEL_365243'].mean() * 100).round(2)
print(f'Part des lignes avec DAYS_EMPLOYED = 365243 : {sentinel_share}%')
display(
    df_app_train_features.groupby('EMPLOYED_SENTINEL_365243')['TARGET']
    .agg(nb_clients='count', taux_defaut='mean')
    .assign(taux_defaut=lambda d: (d['taux_defaut'] * 100).round(2))
)


In [ ]:
cat_cols_focus = [
    'CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE'
]
for col in cat_cols_focus:
    summary = (
        df_app_train_features.groupby(col, dropna=False)['TARGET']
        .agg(nb_clients='count', taux_defaut='mean')
        .sort_values(['taux_defaut', 'nb_clients'], ascending=[False, False])
        .reset_index()
    )
    summary['taux_defaut'] = (summary['taux_defaut'] * 100).round(2)
    print(f'\n### {col}')
    display(summary.head(15))


In [ ]:
ext_source_missing = pd.DataFrame({
    'missing_pct': (df_app_train_features[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isna().mean() * 100).round(2),
    'corr_with_target': df_app_train_features[['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].corr(numeric_only=True)['TARGET'].drop('TARGET').round(3),
})
display(ext_source_missing)
display(df_app_train_features[['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'EXT_SOURCE_MEAN']].corr(numeric_only=True).round(3))


In [ ]:
building_cols = [c for c in df_app_train_features.columns if c.endswith('_AVG') or c.endswith('_MODE') or c.endswith('_MEDI')]
building_summary = pd.DataFrame({
    'missing_pct': (df_app_train_features[building_cols].isna().mean() * 100).round(2),
    'variance': df_app_train_features[building_cols].var(numeric_only=True).round(4),
})
building_summary['family'] = building_summary.index.str.extract(r'^(.*)_(AVG|MODE|MEDI)$')[0]
display(building_summary.sort_values('missing_pct', ascending=False).head(20))
display(building_summary.groupby('family').agg(nb_columns=('family', 'size'), avg_missing_pct=('missing_pct', 'mean')).sort_values('avg_missing_pct', ascending=False).head(20))


In [ ]:
contact_doc_summary = pd.DataFrame({
    'FLAG_PHONE_rate': [round(df_app_train_features['FLAG_PHONE'].mean() * 100, 2)],
    'FLAG_EMAIL_rate': [round(df_app_train_features['FLAG_EMAIL'].mean() * 100, 2)],
    'FLAG_WORK_PHONE_rate': [round(df_app_train_features['FLAG_WORK_PHONE'].mean() * 100, 2)],
    'avg_nb_docs_provided': [round(df_app_train_features['NB_DOCS_PROVIDED'].mean(), 2)],
    'max_nb_docs_provided': [int(df_app_train_features['NB_DOCS_PROVIDED'].max())],
})
display(contact_doc_summary)

doc_flags = [c for c in df_app_train_features.columns if c.startswith('FLAG_DOCUMENT_')]
doc_flag_frequency = (df_app_train_features[doc_flags].mean() * 100).round(3).sort_values()
display(doc_flag_frequency.to_frame('pct_flag_1'))


In [ ]:
bureau_inquiry_cols = [c for c in df_app_train_features.columns if c.startswith('AMT_REQ_CREDIT_BUREAU_')]
bureau_inquiry_summary = pd.DataFrame({
    'missing_pct': (df_app_train_features[bureau_inquiry_cols].isna().mean() * 100).round(2),
    'mean': df_app_train_features[bureau_inquiry_cols].mean().round(3),
    'max': df_app_train_features[bureau_inquiry_cols].max(),
    'corr_with_target': df_app_train_features[['TARGET'] + bureau_inquiry_cols].corr(numeric_only=True)['TARGET'].drop('TARGET').round(3),
})
display(bureau_inquiry_summary.sort_values('corr_with_target', ascending=False))


### Livrables interm?diaires

A l'issue de cette ?tape, on veut formaliser :
- les variables prometteuses pour la mod?lisation ;
- les variables tr?s manquantes, quasi constantes ou douteuses ;
- les premi?res features d?riv?es ? conserver pour la suite ;
- les tables secondaires ? agr?ger en priorit?.
